# Byte Pair Encoding (BPE) — A Beginner's Guide

Byte Pair Encoding (BPE) is a **tokenization algorithm** used by almost every modern language model (GPT-2, GPT-3, GPT-4, LLaMA, RoBERTa, and many more) to break text into smaller pieces called **tokens**.

In this notebook you will:

1. Understand **why** we need tokenization at all.
2. Build a **tiny BPE algorithm from scratch** in plain Python, so you can see exactly what is happening step by step.
3. Use a **real, open-source, pretrained BPE tokenizer** (GPT-2's tokenizer, from Hugging Face) to see BPE in action on real text.

No prior NLP experience is assumed — just basic Python.

## 1. Why do we need tokenization?

Computers don't understand words — they understand numbers. So before we feed text into a machine learning model, we need to convert it into a sequence of numbers.

The simplest ideas are:

- **Character-level**: split text into individual characters (`h`, `e`, `l`, `l`, `o`). Simple, but sequences become very long, and the model has to learn how characters combine into meaning from scratch.
- **Word-level**: split text into whole words (`hello`, `world`). Sequences are short, but the vocabulary becomes huge (every word form counts separately: `run`, `runs`, `running`...), and any word not seen during training becomes an "unknown" token.

**Byte Pair Encoding (BPE)** is a clever middle ground: it learns a vocabulary of *subword* pieces — common words stay whole, while rare or unseen words get broken into smaller, still-meaningful chunks (e.g. `unhappiness` → `un`, `happi`, `ness`).

## 2. How does BPE actually work?

BPE was originally a data-compression algorithm (1994!) before it was repurposed for NLP tokenization. The idea is simple:

1. Start by splitting all words in your training text into individual **characters**. This is your starting vocabulary.
2. Count every pair of **adjacent symbols** that appear next to each other across the whole corpus.
3. Find the **most frequent pair** and merge it into a single new symbol. Add this new symbol to the vocabulary.
4. Repeat steps 2–3 for a fixed number of iterations (or until you reach a target vocabulary size).

Over many iterations, frequently occurring character sequences get merged into larger and larger chunks — common words become single tokens, while rare words remain split into smaller pieces.

Let's implement this from scratch on a tiny toy corpus so you can watch it happen.

### Our worked example

We'll use a tiny corpus of five words with the following frequencies (how many times each word appears in our training text):

| word | frequency |
|---|---|
| hug | 10 |
| pug | 5 |
| pun | 12 |
| bun | 4 |
| hugs | 5 |

BPE training starts by splitting every word into the individual characters used to write it. That gives us our **base vocabulary** — every unique character that appears anywhere in the corpus.

In [1]:
from collections import defaultdict

# Our toy corpus: (word, frequency) pairs
corpus_freqs = {
    "hug": 10,
    "pug": 5,
    "pun": 12,
    "bun": 4,
    "hugs": 5,
}

def word_to_symbols(word):
    return tuple(list(word))  # split into individual characters

# Build a frequency table: {tuple_of_symbols: count}
word_freqs = {word_to_symbols(w): f for w, f in corpus_freqs.items()}

# The base vocabulary is just the set of unique characters seen
base_vocab = sorted({symbol for word in word_freqs for symbol in word})

print("Base vocabulary:", base_vocab)
print()
print("Corpus split into characters:")
for word, freq in word_freqs.items():
    print(f"  {word}  (frequency: {freq})")

Base vocabulary: ['b', 'g', 'h', 'n', 'p', 's', 'u']

Corpus split into characters:
  ('h', 'u', 'g')  (frequency: 10)
  ('p', 'u', 'g')  (frequency: 5)
  ('p', 'u', 'n')  (frequency: 12)
  ('b', 'u', 'n')  (frequency: 4)
  ('h', 'u', 'g', 's')  (frequency: 5)


This matches the base vocabulary `["b", "g", "h", "n", "p", "s", "u"]` and the character-split corpus: `hug` -> 10 times, `pug` -> 5, `pun` -> 12, `bun` -> 4, `hugs` -> 5.

In [2]:
def get_pair_counts(word_freqs):
    """Count how often each adjacent pair of symbols occurs, across the whole corpus."""
    pair_counts = defaultdict(int)
    for word, freq in word_freqs.items():
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            pair_counts[pair] += freq
    return pair_counts

pair_counts = get_pair_counts(word_freqs)

print("All adjacent pairs and their counts:")
for pair, count in sorted(pair_counts.items(), key=lambda x: -x[1]):
    print(f"  {pair}  ->  {count} times")

All adjacent pairs and their counts:
  ('u', 'g')  ->  20 times
  ('p', 'u')  ->  17 times
  ('u', 'n')  ->  16 times
  ('h', 'u')  ->  15 times
  ('g', 's')  ->  5 times
  ('b', 'u')  ->  4 times


Look at the pair `("u", "g")`: it shows up in `hug`, `pug`, and `hugs`, for a combined total of **10 + 5 + 5 = 20** occurrences — the most frequent pair in the corpus. That means it will be the very first merge rule BPE learns.

In [6]:
def merge_pair(pair, word_freqs):
    """Merge every occurrence of `pair` into a single new symbol."""
    new_word_freqs = {}
    a, b = pair
    merged_symbol = a + b

    for word, freq in word_freqs.items():
        new_word = []
        i = 0
        while i < len(word):
            # If we find the pair right here, merge it into one symbol
            if i < len(word) - 1 and word[i] == a and word[i + 1] == b:
                new_word.append(merged_symbol)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_word_freqs[tuple(new_word)] = freq

    return new_word_freqs

In [7]:
def train_bpe(word_freqs, num_merges, verbose=True):
    word_freqs = dict(word_freqs)
    merges = []  # the ordered list of merge rules we learn, in the order learned

    for step in range(num_merges):
        pair_counts = get_pair_counts(word_freqs)
        if not pair_counts:
            break

        # Pick the single most frequent pair
        best_pair = max(pair_counts, key=pair_counts.get)
        best_count = pair_counts[best_pair]

        word_freqs = merge_pair(best_pair, word_freqs)
        merges.append(best_pair)

        if verbose:
            merged_token = "".join(best_pair)
            print(f"Merge rule {step + 1}: {best_pair} -> '{merged_token}'  "
                  f"(occurred {best_count} times)")

    return word_freqs, merges

final_word_freqs, learned_merges = train_bpe(word_freqs, num_merges=3)

Merge rule 1: ('u', 'g') -> 'ug'  (occurred 20 times)
Merge rule 2: ('u', 'n') -> 'un'  (occurred 16 times)
Merge rule 3: ('h', 'ug') -> 'hug'  (occurred 15 times)


Let's trace through exactly what happens, step by step:

1. **Merge 1**: `("u", "g") -> "ug"` (20 occurrences). Vocabulary becomes `["b","g","h","n","p","s","u","ug"]`. The corpus is now: `hug`->10, `pug`->5, `pun`->12, `bun`->4, `hugs`->5, all rewritten with `ug` merged wherever it appears.
2. **Merge 2**: now `("h","ug")` occurs 15 times (in `hug` and `hugs`), but `("u","n")` occurs **16** times (in `pun` and `bun`) — so `("u","n") -> "un"` wins this round, even though `hug`-related pairs looked promising. Vocabulary becomes `["b","g","h","n","p","s","u","ug","un"]`.
3. **Merge 3**: now `("h","ug")` (15 occurrences) is the most frequent remaining pair, so it merges into `"hug"`. Vocabulary becomes `["b","g","h","n","p","s","u","ug","un","hug"]` — our first three-letter token!

This is a great lesson: BPE is **greedy and frequency-driven**. It doesn't know or care about English morphology — it just always merges whatever pair is statistically most common at each step, even if that means `un` gets merged before `hug` does.

In [8]:
print("Final tokenized forms of each word in our corpus:\n")
for word, freq in final_word_freqs.items():
    print(f"  {' '.join(word):20s} (frequency: {freq})")

print("\nLearned merge rules, in the order they were learned:")
for i, pair in enumerate(learned_merges, 1):
    print(f"  {i}. {pair[0]!r} + {pair[1]!r} -> {''.join(pair)!r}")

full_vocab = base_vocab + ["".join(p) for p in learned_merges]
print("\nFull vocabulary after training:", full_vocab)

Final tokenized forms of each word in our corpus:

  hug                  (frequency: 10)
  p ug                 (frequency: 5)
  p un                 (frequency: 12)
  b un                 (frequency: 4)
  hug s                (frequency: 5)

Learned merge rules, in the order they were learned:
  1. 'u' + 'g' -> 'ug'
  2. 'u' + 'n' -> 'un'
  3. 'h' + 'ug' -> 'hug'

Full vocabulary after training: ['b', 'g', 'h', 'n', 'p', 's', 'u', 'ug', 'un', 'hug']


### Applying learned merges to brand-new words

Once BPE has learned its merge rules, tokenizing a new word means: split it into characters, then apply each learned merge rule **in the order it was learned**. But what happens if a word contains a character that was never seen during training (like `m` or `t`, which never appeared in our corpus)? Any character outside the base vocabulary gets replaced with a special `[UNK]` ("unknown") token.

In [9]:
UNK_TOKEN = "[UNK]"

def tokenize_word(word, merges, base_vocab):
    # Step 1: split into characters, replacing anything outside the
    # base vocabulary with [UNK]
    symbols = [ch if ch in base_vocab else UNK_TOKEN for ch in word]

    # Step 2: apply every learned merge rule, in order
    for pair in merges:
        a, b = pair
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                new_symbols.append(a + b)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols

    return symbols

for test_word in ["bug", "mug", "thug"]:
    print(f"{test_word!r:8s} -> {tokenize_word(test_word, learned_merges, base_vocab)}")

'bug'    -> ['b', 'ug']
'mug'    -> ['[UNK]', 'ug']
'thug'   -> ['[UNK]', 'hug']


This matches the walkthrough exactly:

- `"bug"` -> `["b", "ug"]` — every character (`b`, `u`, `g`) was in the base vocabulary, and the `(u, g) -> ug` merge rule applies.
- `"mug"` -> `["[UNK]", "ug"]` — the letter `m` was never seen during training, so it becomes `[UNK]`, but `u` and `g` still merge normally.
- `"thug"` -> `["[UNK]", "hug"]` — `t` becomes `[UNK]`; meanwhile `u`+`g` merge into `ug`, and then `h`+`ug` merge into `hug`, exactly following the order the merge rules were learned.

This shows an important limitation of plain BPE: it can only represent characters (and combinations of characters) it saw during training. Anything truly novel becomes `[UNK]` — which is one reason real tokenizers train on *huge, diverse* corpora, and often operate on raw *bytes* rather than Unicode characters, so that in practice nothing is ever truly unrepresentable.

## 3. BPE in the real world: Hugging Face `tokenizers`

Our toy example used 15 words and 8 merges. Real tokenizers (like GPT-2's) are trained on **billions of words** with tens of thousands of merges, producing a vocabulary of tens of thousands of subword tokens.

Let's use the open-source [`tokenizers`](https://github.com/huggingface/tokenizers) library from Hugging Face to train a *real* BPE tokenizer on a small sample text, so you can see the same algorithm you just wrote by hand, scaled up and production-ready.

In [10]:
# Install the open-source Hugging Face tokenizers library
!pip install tokenizers --quiet

In [11]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# A slightly larger sample text to train on
sample_text = """
The quick brown fox jumps over the lazy dog.
Byte pair encoding is a subword tokenization algorithm.
Tokenization turns text into tokens that a model can understand.
The lazy dog was not amused by the quick brown fox.
Subword tokenizers handle rare and unseen words gracefully.
"""

with open("sample.txt", "w") as f:
    f.write(sample_text)

# 1. Create a tokenizer with an (empty, untrained) BPE model
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

# 2. Split on whitespace before applying BPE merges (same idea as our </w> marker)
tokenizer.pre_tokenizer = Whitespace()

# 3. Configure a trainer: target vocabulary size and special tokens
trainer = BpeTrainer(vocab_size=200, special_tokens=["[UNK]", "[PAD]"])

# 4. Train BPE on our sample text -- this runs the exact algorithm we built by hand,
#    just far more optimized in Rust under the hood
tokenizer.train(["sample.txt"], trainer)

print("Learned vocabulary size:", tokenizer.get_vocab_size())




Learned vocabulary size: 149


In [12]:
# Try tokenizing a sentence, including a word never seen during training
output = tokenizer.encode("The quick foxes are jumping unbelievably fast")
print("Tokens:", output.tokens)
print("Token IDs:", output.ids)

Tokens: ['The', 'quick', 'fox', 'e', 's', 'a', 're', 'jump', 'ing', 'u', 'n', 'b', 'el', 'i', 'e', 'v', 'a', 'b', 'l', 'y', 'f', 'as', 't']
Token IDs: [45, 72, 70, 10, 24, 6, 105, 136, 123, 26, 19, 7, 88, 14, 10, 27, 6, 7, 17, 30, 11, 81, 25]


Even though `foxes`, `jumping`, and `unbelievably` never appeared (in that exact form) in our training text, the tokenizer can still represent them by combining known subword pieces — exactly like our toy example did with `lower`.

## 4. BPE inside a real pretrained model: GPT-2's tokenizer

Finally, let's look at the actual BPE tokenizer used by **GPT-2**, an open-source model released by OpenAI. This tokenizer was trained on a huge web-text corpus and is the same one used to prepare text for GPT-2 (and it heavily influenced GPT-3/4's tokenizer too). We access it through the open-source [`transformers`](https://github.com/huggingface/transformers) library.

In [ ]:
!pip install transformers --quiet

In [ ]:
from transformers import GPT2Tokenizer

gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

sentences = [
    "Byte pair encoding is everywhere in modern NLP.",
    "Tokenization strategies affect how well models handle rare words.",
    "supercalifragilisticexpialidocious",
]

for sentence in sentences:
    tokens = gpt2_tokenizer.tokenize(sentence)
    ids = gpt2_tokenizer.encode(sentence)
    print(f"Text:   {sentence}")
    print(f"Tokens: {tokens}")
    print(f"IDs:    {ids}")
    print("-" * 60)

A few things to notice:

- Common whole words (like `Tokenization`) often become a **single token**.
- Rare or unusual words (like `supercalifragilisticexpialidocious`) get **split into several subword pieces**.
- You may see tokens starting with a special character (often shown as `Ġ` in GPT-2) — this marks "a space came before this token." GPT-2's BPE operates on raw bytes, so it needs a way to represent spaces as part of tokens, which is a small variation on the character-level BPE we built by hand.

## 5. Summary

| Concept | What we did |
|---|---|
| **Why tokenize?** | Convert text to numbers efficiently, balancing vocabulary size and sequence length |
| **BPE algorithm** | Start from characters, repeatedly merge the most frequent adjacent pair |
| **From scratch** | Implemented `get_pair_counts`, `merge_pair`, and a training loop in plain Python |
| **Generalization** | Learned merges apply to unseen words by reusing the same merge rules |
| **Real-world tools** | Trained a real BPE tokenizer with Hugging Face `tokenizers` |
| **Pretrained model** | Inspected GPT-2's actual open-source BPE tokenizer from `transformers` |

### Key takeaway
BPE is simple: **count adjacent pairs, merge the most frequent one, repeat.** That one idea, run millions of times over huge datasets, is what turns raw text into the tokens that power today's large language models.

### Try it yourself
- Change `num_merges` in the from-scratch example and see how the tokenization changes.
- Change `vocab_size` in the Hugging Face trainer and observe how larger vocabularies produce fewer, longer tokens per word.
- Try tokenizing words in a language other than English with the GPT-2 tokenizer — what do you notice?